<a href="https://colab.research.google.com/github/alanna-jc/VAE_Anomaly_Detector/blob/main/VAE_Anomaly_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

VAE Anomaly Detector

In [ ]:
# Download the dataset, setup packages
import os
import cv2
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

if not os.path.exists('dataset.zip'):
  !gdown 1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
  !unzip -q -u dataset.zip
else:
  print('Already downloaded')

Downloading...
From: https://drive.google.com/uc?id=1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
To: /content/dataset.zip
100% 2.11M/2.11M [00:00<00:00, 90.3MB/s]


In [ ]:
# Some helper functions
def load_dataset(class_name = 'pasta'):
  assert class_name in ['pasta', 'screws']
  dir = './dataset/'+class_name+'/'
  training_images = []
  testing_images = []
  testing_labels = []
  for file_name in os.listdir(dir+'train/good/'):
    training_images.append(cv2.cvtColor(cv2.imread(dir+'train/good/'+file_name), cv2.COLOR_BGR2RGB))
  for file_name in os.listdir(dir+'test/good/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/good/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(0)
  for file_name in os.listdir(dir+'test/bad/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/bad/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(1)

  # returns a normalized (0-1) numpy array of size (n,)
  return np.array(training_images)/255., np.array(testing_images)/255., np.array(testing_labels)

def basic_evaluation(predictions : np.ndarray, targets : np.ndarray):
  print(targets)
  print(predictions)
  print('AUROC Score:', roc_auc_score(targets, predictions))

In [ ]:
# TODO import any packages here

import torch #will use pytorch for convolutional autoencoder
import torch.nn as nn
import torch.optim as optim


In [ ]:
#define class for autoencoder module
#have it inherit from nm.Module class
class VAE_Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        # encoder has 3 convolutional layers which reduces dimensions in half each layer (stride = 2)
        # these layers will act as feature extractors
        # each layer is 3x3
        self.encoder = nn.Sequential(
            # (in_channels, out_channels, kernel_size, stride, padding)
            nn.Conv2d(3,16,3,stride=2,padding=1),
            nn.ReLU(),
            nn.Conv2d(16,32,3,stride=2,padding=1),
            nn.ReLU(),
            nn.Conv2d(32,64,3,stride=2,padding=1),
            nn.ReLU()
        )

    def forward(self,x):
        z = self.encoder(x)
        return z

In [ ]:
#define class for autoencoder module
#have it inherit from nm.Module class
class VAE_Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        #decoder mirrors encoder and reverses to reconstruct the spacial resolution
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64,32,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,16,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16,3,3,stride=2,padding=1,output_padding=1),
            nn.Sigmoid()
        )

    def forward(self,x):
        x_hat = self.decoder(z)
        return x_hat

In [ ]:
# TODO This class will be the class to modify.
# The current implementation of this model tries to take the median of all the
# Training images and then measure the L2 distance between this median and any
# test images.

class AnomalyDetector:

  def __init__(self):
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # change this for separated code
    self.model = ConvolutionalAutoencoder().to(self.device)

  def create_model(self, dataset):

    # convert numpy to torch, need to rearrange the dimensions
    x = torch.tensor(dataset).permute(0,3,1,2).float().to(self.device)

    #use default value of lr=1e-3 for learning rate for optimizer
    optimizer = optim.Adam(self.model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    epochs = 20

    for epoch in range(epochs):

      self.model.train()

      #reset gradients to zero
      optimizer.zero_grad()

      x_hat = self.model(x)

      loss = loss_fn(x_hat, x)

      loss.backward()

      optimizer.step()

      print("epoch",epoch,"loss",loss.item())


  def predict(self, test_data):

    self.model.eval()

    x = torch.tensor(test_data).permute(0,3,1,2).float().to(self.device)

    with torch.no_grad():

      x_hat = self.model(x)

      errors = torch.mean((x - x_hat)**2, dim=(1,2,3))

    return errors.cpu().numpy()

  # def __init__(self,):
  #   self.model = None
  # def create_model(self, dataset : np.ndarray):
  #   self.model = np.mean(dataset, axis=0)
  #   print(dataset.shape)
  #   plt.figure(figsize = (16,5))
  #   plt.imshow(self.model)
  #   plt.show()
  # def predict(self, test_data : np.ndarray):
  #   return np.mean(np.square(test_data - self.model), axis=(1,2,3))



In [ ]:
# TODO use class above as well as helper functions to generate
# predictions on the datasets and evaluate the results.

def do_analysis(ad, class_name):
  training_images, testing_images, testing_labels = load_dataset(class_name=class_name)
  ad.create_model(training_images)
  predictions = ad.predict(testing_images)
  basic_evaluation(predictions, testing_labels)

print("screws analysis")
do_analysis(AnomalyDetector(), 'screws')
print("pasta analysis")
do_analysis(AnomalyDetector(), 'pasta')


screws analysis
epoch 0 loss 0.08357822895050049
epoch 1 loss 0.08330173045396805
epoch 2 loss 0.08301235735416412
epoch 3 loss 0.08270282298326492
epoch 4 loss 0.08235737681388855
epoch 5 loss 0.0819530189037323
epoch 6 loss 0.08146560937166214
epoch 7 loss 0.0808614045381546
epoch 8 loss 0.0800987035036087
epoch 9 loss 0.07912652939558029
epoch 10 loss 0.07789688557386398
epoch 11 loss 0.07638102769851685
epoch 12 loss 0.07459795475006104
epoch 13 loss 0.07268904894590378
epoch 14 loss 0.07095547765493393
epoch 15 loss 0.06981078535318375
epoch 16 loss 0.06914742290973663
epoch 17 loss 0.06832282245159149
epoch 18 loss 0.06702879071235657
epoch 19 loss 0.0653478279709816
[0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
[0.06383224 0.06161762 0.0599402  0.06199336 0.06010342 0.06757762
 0.06339963 0.05485952 0.05891377 0.06088537 0.06128211 0.06420222
 0.05931539 0.06284233 0.06154652]
AUROC Score: 0.48
pasta analysis
epoch 0 loss 0.08541227877140045
epoch 1 loss 0.08503616601228714
epoch 2 loss 0.084